# Dump Test Metrics Notebook

- Source: `scripts/dump_test_metrics.py`
- 목적: 원본 파이썬 파일을 단계별로 실행/설명하기 위한 노트북 버전
- 실행 방법: 위에서 아래로 순서대로 실행


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # 노트북이 다른 경로에서 열렸을 때 프로젝트 루트 자동 탐색
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'src').exists() and (parent / 'configs').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


## Step 1. Setup and Imports

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
"""두 태스크(로스팅/결점두) 테스트 지표를 docs/test_metrics.json에 저장하는 스크립트.

인 파일 (docs/test_metrics.json)은 Streamlit 앱 '성능 리포트' 탭에서
 KPI 카드와 그래프를 그리는 데 사용된다.

다음 상황에서 반드시 다시 실행해야 한다:
- 모델을 재학습한 후 (모델 교체됨)
- 테스트 스플릿 CSV가 바뀐 후 (데이터 교체)

사용 예시::

    python scripts/dump_test_metrics.py

생성 예시 (docs/test_metrics.json) ::

    {
      "roast":  {"test_acc": 0.94, "test_macro_f1": 0.94, "per_class_f1": {...}},
      "defect": {"test_acc": 0.78, "test_macro_f1": 0.79, "per_class_f1": {...}}
    }
"""
from __future__ import annotations
import json
from pathlib import Path

import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import DataLoader

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.dataset import SingleTaskDataset, get_transforms
from src.model import CoffeeClassifier


## Step 2. Function: eval_one

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def eval_one(cfg_path: str, ckpt_path: str) -> dict:
    cfg = load_config(cfg_path)
    set_seed(cfg["seed"])
    device = "cuda" if torch.cuda.is_available() else "cpu"

    task = cfg["task"]
    classes = cfg["classes"]
    c2i = {c: i for i, c in enumerate(classes)}

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    backbone = (ckpt.get("config") or cfg)["model"]["name"]

    model = CoffeeClassifier(
        backbone=backbone, n_classes=len(classes),
        pretrained=False, dropout=cfg["model"]["dropout"],
        hidden=cfg["model"]["hidden"],
    ).to(device)
    model.load_state_dict(state)
    model.eval()

    ds = SingleTaskDataset(
        csv_path=cfg["data"]["test_csv"], task=task, class_to_idx=c2i,
        transform=get_transforms(task=task, train=False,
                                 img_size=cfg["data"]["img_size"]),
    )
    loader = DataLoader(ds, batch_size=cfg["train"]["batch_size"],
                        shuffle=False, num_workers=cfg["train"]["num_workers"])

    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            ps.append(model(x.to(device)).argmax(1).cpu())
            ys.append(y)
    y_true = torch.cat(ys).numpy()
    y_pred = torch.cat(ps).numpy()

    acc = float(accuracy_score(y_true, y_pred))
    f1 = float(f1_score(y_true, y_pred, average="macro"))
    report = classification_report(
        y_true, y_pred, target_names=classes, digits=4, output_dict=True,
        zero_division=0,
    )
    return {
        "task": task,
        "n_classes": len(classes),
        "n_test": int(len(y_true)),
        "test_acc": acc,
        "test_macro_f1": f1,
        "ckpt_val_macro_f1": float(ckpt.get("val_macro_f1", -1)),
        "ckpt_epoch": int(ckpt.get("epoch", -1)),
        "classes": classes,
        "per_class_f1": {c: float(report[c]["f1-score"]) for c in classes},
    }


## Step 3. Function: main

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def main() -> None:
    out = {
        "roast": eval_one("configs/default.yaml", "checkpoints/best_roast.pth"),
        "defect": eval_one("configs/defect.yaml", "checkpoints/best_defect.pth"),
    }
    Path("docs").mkdir(exist_ok=True)
    Path("docs/test_metrics.json").write_text(
        json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(json.dumps(out, indent=2, ensure_ascii=False))


## Step 4. Run Entry Point

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
if __name__ == "__main__":
    main()


## 실행 파라미터 가이드

- 이 파일은 원래 CLI 인자(argparse) 기반으로 동작합니다.
- 노트북에서는 인자 대신 아래처럼 변수 셀을 만들어 실행하세요.


In [ ]:
# 예시 파라미터 셀
CONFIG_PATH = 'configs/default.yaml'
CKPT_PATH = None
